# Solutions – Day 44

In [ ]:
# Exercise 1: vLLM server with TinyLlama
tiny_llama_cmd = """
python -m vllm.entrypoints.api_server \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0
"""
print(tiny_llama_cmd)

# After server is running:
import requests
prompts = ["Hello", "Explain AI", "Tell a joke", "What is love?", "Write a poem."]
for p in prompts:
    resp = requests.post("http://localhost:8000/generate", json={"prompt": p, "max_tokens": 50})
    print(resp.json().get("text", ""))

In [ ]:
# Exercise 2: Throughput comparison
import time
from vllm import LLM, SamplingParams
from transformers import pipeline

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
prompts = ["Write a sentence about cats."] * 20

# HF
hf = pipeline("text-generation", model=model_id, device=0)
start = time.time()
for p in prompts:
    hf(p, max_new_tokens=20)
hf_t = time.time() - start

# vLLM
llm = LLM(model=model_id)
params = SamplingParams(max_tokens=20)
start = time.time()
llm.generate(prompts, params)
vllm_t = time.time() - start

print(f"HF: {hf_t:.2f}s, vLLM: {vllm_t:.2f}s, speedup: {hf_t/vllm_t:.2f}x")

In [ ]:
# Exercise 3: Parameter tuning
params = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=200)
output = llm.generate(["Tell a story about a dragon."], params)
print(output[0].outputs[0].text)

In [ ]:
# Exercise 4: TGI request (after docker run)
import requests
response = requests.post('http://localhost:8080/generate', json={'inputs': 'Hello', 'parameters': {'max_new_tokens': 50}})
print(response.json().get('generated_text', ''))

In [ ]:
# Exercise 5: Optimise your project – pseudo code example
# Replace
#   response = openai.ChatCompletion.create(...)
# with local vLLM call or self-hosted endpoint.
# Use vLLM for RAG context generation.
print("Integrate vLLM into your RAG pipeline for sub‑second latency.")